In [1]:
import tsim
import time

In [2]:
c = tsim.Circuit.from_file("Surface3d.stim")

In [4]:

# Compilation step, takes about 3s
sampler = c.compile_detector_sampler(strategy="cutting")
start_time = time.perf_counter()
n = 10_000  # On GPU, increase batch size to fully utilize VRAM
sampler.sample(shots=n, batch_size=n)
end_time = time.perf_counter()
print(f"Time per shot: {(end_time - start_time) / n * 1e6:.2f} microseconds")

Time per shot: 355.67 microseconds


In [6]:
from tsim import Circuit
c.diagram(height=150)
c.diagram("pyzx");
sampler = c.compile_sampler()
sampler.sample(shots=100_000)
detector_sampler = c.compile_detector_sampler()
detector_sampler.sample(shots=10, append_observables=True)


array([[False, False, False, False, False,  True, False,  True, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False],
       [False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False],
       [False, False, False, False, False, False, False, False, False,
        False, Fa

In [11]:
c.diagram(height=800)

In [7]:
detector_bits, observable_bits = detector_sampler.sample(
    shots=100_000, separate_observables=True
)
assert not detector_bits.any()
observable_bits

AssertionError: 

In [12]:
# Per gestire il rumore, compiliamo un decodificatore.
# Questo analizza i "detection events" e predice la correzione necessaria.
decoder = c.compile_decoder()

# Usiamo il decodificatore per predire la correzione e contare gli errori logici.
num_errors, _ = decoder.decode(shots=100_000, date=detector_bits)

print(f"Numero di errori logici: {num_errors}")
print(f"Tasso di errore logico: {num_errors / 100_000}")

AttributeError: 'Circuit' object has no attribute 'compile_decoder'

In [13]:
import stim
p = 0.0001
stim_circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=2,
    distance=3,
    before_round_data_depolarization=p,
    before_measure_flip_probability=p,
    after_clifford_depolarization=p,
    after_reset_flip_probability=p,
)
circuit = Circuit.from_stim_program(stim_circuit)
sampler = circuit.compile_detector_sampler()

stim_sampler = stim_circuit.compile_detector_sampler()
circuit.diagram("pyzx-dets")
n = 1_000_000
samples = sampler.sample(n)
samples
stim_samples = stim_sampler.sample(n)
stim_samples

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]],
      shape=(1000000, 16))

In [14]:
circuit.diagram(height=800)